In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

behavior_model = ProductionBehaviorModel(num_classes=3)
behavior_model.load_state_dict(
    torch.load("/content/drive/MyDrive/behavior_model_final.pth", map_location=DEVICE)
)
behavior_model.to(DEVICE)
behavior_model.eval()

CLASS_NAMES = {0: "Awake", 1: "Sleeping", 2: "Talking"}
CLASS_COLORS = {0: (0, 255, 0), 1: (0, 0, 255), 2: (255, 0, 0)}

img_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=5,
    refine_landmarks=True
)

tracker = DeepSort(max_age=30)
yolo_model = YOLO("/content/drive/MyDrive/YOLO/YOLOv9face.pt")

def predict_behavior(face_bgr, landmark_vec):
    face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    img_tensor = img_transform(face_rgb).unsqueeze(0).to(DEVICE)
    lm_tensor  = torch.tensor(landmark_vec, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = behavior_model(img_tensor, lm_tensor)
        pred   = torch.argmax(logits, dim=1).item()
    return pred

cap = cv2.VideoCapture("/content/drive/MyDrive/YOLO/talk.mp4")
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    '/content/output_j.mp4',
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

SMOOTH_WINDOW  = 10   # majority vote over last N frames
SLEEP_FRAMES   = 45   # consecutive sleeping predictions before alert

student_data   = {}   # track_id -> {total_frames, sleep_frames}
pred_history   = {}   # track_id -> deque of recent raw predictions
sleep_counter  = {}   # track_id -> consecutive sleeping frame count

from collections import deque

while True:
    ret, frame = cap.read()
    if not ret:
        break

    sleeping_students = 0

    # YOLO detection
    results = yolo_model(frame, imgsz=640, conf=0.6)
    detections = []
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            if conf > 0.6:
                detections.append(([x1, y1, x2-x1, y2-y1], conf, 'face'))

    tracks = tracker.update_tracks(detections, frame=frame)
    confirmed_tracks = [t for t in tracks if t.is_confirmed()]
    total_students   = len(confirmed_tracks)

    for track in confirmed_tracks:
        track_id = track.track_id

        # Init state for new track
        if track_id not in student_data:
            student_data[track_id]  = {"total_frames": 0, "sleep_frames": 0}
            pred_history[track_id]  = deque(maxlen=SMOOTH_WINDOW)
            sleep_counter[track_id] = 0

        student_data[track_id]["total_frames"] += 1

        l, t, r, b = map(int, track.to_ltrb())
        h_frame, w_frame, _ = frame.shape
        l = max(0, l); t = max(0, t)
        r = min(w_frame, r); b = min(h_frame, b)

        face_bgr = frame[t:b, l:r]
        if face_bgr.shape[0] < 60 or face_bgr.shape[1] < 60:
            continue

        # MediaPipe on face crop
        face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
        mp_result = face_mesh.process(face_rgb)

        if not mp_result.multi_face_landmarks:
            # No landmarks detected — reuse last prediction if available
            if len(pred_history[track_id]) == 0:
                continue
            # else fall through to smoothed status below using existing history
            raw_pred = pred_history[track_id][-1]
        else:
            lm_vec   = extract_landmarks(face_rgb, mp_result.multi_face_landmarks[0].landmark)
            raw_pred = predict_behavior(face_bgr, lm_vec)

        pred_history[track_id].append(raw_pred)

        # Majority vote smoothing
        history = list(pred_history[track_id])
        smoothed_pred = max(set(history), key=history.count)

        # Consecutive sleep counter — for alert threshold
        if smoothed_pred == 1:
            sleep_counter[track_id] += 1
        else:
            sleep_counter[track_id] = max(0, sleep_counter[track_id] - 1)

        # Final status
        if smoothed_pred == 1 and sleep_counter[track_id] > SLEEP_FRAMES:
            status = "Sleeping"
            sleeping_students += 1
            student_data[track_id]["sleep_frames"] += 1
        elif smoothed_pred == 1:
            status = "Drowsy"     # model says sleeping but not long enough yet
        else:
            status = CLASS_NAMES[smoothed_pred]

        color = CLASS_COLORS.get(smoothed_pred, (0, 255, 0))
        if status == "Drowsy":
            color = (0, 255, 255)

        cv2.rectangle(frame, (l, t), (r, b), color, 2)
        cv2.putText(frame,
                    f"ID:{track_id} {status}",
                    (l, t - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, color, 2)

    # HUD
    sleep_percent = (sleeping_students / total_students * 100) if total_students > 0 else 0
    cv2.putText(frame, f"Sleep %: {sleep_percent:.2f}",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    if sleep_percent > 30:
        cv2.putText(frame, "ALERT: Many students sleeping!",
                    (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

    out.write(frame)

cap.release()
out.release()

import csv

with open("/content/sleep_report_out.csv", mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Student ID", "Total Time (sec)", "Sleep Time (sec)", "Sleep Percentage"])
    for track_id, data in student_data.items():
        total_time = data["total_frames"] / fps
        sleep_time = data["sleep_frames"] / fps
        sp = (sleep_time / total_time * 100) if total_time > 0 else 0
        writer.writerow([track_id, round(total_time, 2), round(sleep_time, 2), round(sp, 2)])

print("Done.")

NameError: name 'nn' is not defined